![Platform+](images/banner.png)

# LAB 5: APPLY AI TO TEXT
_Hands-on Snowflake for data practitioners_

> **Persona track: Data Engineer** — same lab, your angle.

👉 **In this lesson, we'll build an AI-powered enrichment pipeline — standardize, extract, embed — then operationalize it with a Dynamic Table.**

📚 **By the end you'll be able to:**
- Standardize inconsistent text into a target format with AI_COMPLETE
- Extract key-value fields from descriptions with AI_EXTRACT
- Translate text columns inline with AI_TRANSLATE
- Create 768-dim embeddings for similarity search with AI_EMBED
- Operationalize incremental AI enrichment with a Dynamic Table

**Time:** ~35 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
%%sql -r dataframe_1
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

> 📌 **Why you can call these AI functions:** access to Cortex AI functions is granted through the [`SNOWFLAKE.CORTEX_USER`](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access) **database role**, already granted to your role during setup.
>
> A **database role** is a role whose privileges are scoped to a single database — here, the built-in `SNOWFLAKE` database. Unlike an account role, a database role can't be activated directly in a session; it's *granted to* your account role (`{{my_role}}`), which then inherits its privileges. It's Snowflake's standard way to package up "the privileges needed to use feature X" and hand them to whoever should have them. More on [database roles](https://docs.snowflake.com/en/user-guide/security-access-control-overview). Everything you build in this lab still lands in your own database.

## Snowflake AI Functions — the big picture
Snowflake gives you a whole **family of AI functions** (part of *Cortex AISQL*) that run large language models for you, right inside the database. Here's the magic: **you don't change how you work.** No model to train, no GPUs to provision, no data to ship out to an external API. You call them like any other function — in **SQL or Python** — on data that never leaves Snowflake. For a data engineer, that means AI becomes just another transformation step in your pipeline.

There are many of these functions. In this lab you'll chain five into an enrichment pipeline:

| Function | What it does |
|---|---|
| `AI_COMPLETE` | Generates/rewrites text from a prompt (standardization) |
| `AI_EXTRACT` | Pulls named fields out of free text into a structured object |
| `AI_TRANSLATE` | Converts text between languages |
| `AI_EMBED` | Turns text into a vector for similarity search |
| `AI_COUNT_TOKENS` | Estimates token cost before you schedule a pipeline |

👉 Browse the full list any time in the [Cortex AI Functions docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql).

## 1 · Explore the Raw Product Data
The `PRODUCTS` table has a `description` column with free-text from multiple suppliers. Each supplier formats differently — some include origin, roast level, and flavor notes inline; others are terse. Your job is to **standardize** them and add AI-derived structure that downstream consumers can query cheaply.

In [ ]:
%%sql -r dataframe_2
-- Names, categories, prices — plus the raw supplier description
SELECT product_id, product_name, category, unit_price, description
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
ORDER BY product_id;

> 🔎 **Review the output.** Notice the descriptions vary in style and detail. The next steps use AI to standardize them and pull out structured fields.

## 2 · Standardize Descriptions With AI_COMPLETE
[`AI_COMPLETE`](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) is the **general-purpose** function in the family. The others are *task-specific* — AI_EXTRACT pulls fields, AI_TRANSLATE translates, AI_SENTIMENT scores tone — but AI_COMPLETE takes **any prompt you write** and returns generated text, so it can tackle anything you can describe in words. It's also the one where **you choose the model** to run on (its first argument), letting you trade cost, speed, and quality — the note below says more.

Here we use only its **simplest form** — one prompt string plus your data — to **rewrite** inconsistent supplier text into a consistent 2-sentence format. (AI_COMPLETE can also do multi-turn chat, structured JSON output, and even images; we won't need those today.)

In [ ]:
%%sql -r dataframe_3
SELECT product_name,
       AI_COMPLETE(
           'llama3.1-8b',
           'Rewrite this coffee-shop product description in exactly two sentences. '
           || 'Sentence 1: what it is. Sentence 2: a flavor or use recommendation. '
           || 'Do not include any preamble, introduction, or extra commentary. '
           || 'Respond with only the two sentences. '
           || 'Description: ' || description
       ) AS standardized_description
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
WHERE description IS NOT NULL
LIMIT 5;

> 📌 **Note:** **Engineering insight:** By pinning the model (`llama3.1-8b`) you control cost and latency. Use small models for formatting/standardization; reserve large models for nuanced work.

## 3 · Extract Structured Fields With AI_EXTRACT
[`AI_EXTRACT`](https://docs.snowflake.com/en/sql-reference/functions/ai_extract) pulls named fields from text and returns a structured OBJECT under a `response` key.

> 📎 **A quick note on the syntax.** AI_EXTRACT returns a structured object with everything nested under a `response` key. You read into it with a colon *path*: in the query below, **`):response::STRING`** steps into that `response` key and `::STRING` renders the whole result as text so you can eyeball it. To pull a *single* field you'd go one level deeper — `:response:origin_country::STRING` — which is exactly what you'll do when you build the enrichment table in §6.

🧩 **Your turn.** AI_EXTRACT takes the text plus a **list of the fields you want**. The cell below is missing that list. From each description we want three fields — `origin_country`, `roast_level`, and `flavor_notes`. Write them as an array in the `______` blank, then run it. Stuck? Reveal the reference below the cell.

In [ ]:
%%sql -r dataframe_4
SELECT product_name,
       AI_EXTRACT(
           description,
           ______   -- 🧩 fill in the array of fields to extract
       ):response::STRING AS extract_result_response
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
WHERE description IS NOT NULL
LIMIT 5;

> <details><summary>🧩 Stuck? Reference solution</summary>
>
>     ['origin_country', 'roast_level', 'flavor_notes']
>
> AI_EXTRACT takes an array of the field names you want pulled from the text; it returns each under the `response` key.
> </details>

> 🔎 **Review the output.** `extract_result_response` is the object of fields AI_EXTRACT pulled — `origin_country`, `roast_level`, `flavor_notes` — rendered as text. Fields the description doesn't mention come back null (correct behavior). In §6 you'll reach into individual fields with the deeper path (`:response:origin_country`).

## 4 · Translate Multilingual Feedback With AI_TRANSLATE
[`AI_TRANSLATE`](https://docs.snowflake.com/en/sql-reference/functions/ai_translate) converts text between languages so your pipeline works in a single language.

Here we demo English → Spanish — in production you'd detect the language first and only translate the rows that need it.

In [ ]:
%%sql -r dataframe_5
-- AI_TRANSLATE(source_text, source_lang, target_lang); '' source = auto-detect
SELECT feedback_id,
       LEFT(comment, 60) AS original_comment,
       AI_TRANSLATE(comment, '', 'es') AS spanish_translation
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL
LIMIT 5;

> 📌 **Note:** **Pipeline tip:** add a `detected_language` column and only translate rows where it isn't already your target language — saves tokens and time.

## 5 · Generate Embeddings With AI_EMBED
[`AI_EMBED`](https://docs.snowflake.com/en/sql-reference/functions/ai_embed) converts text into a numeric vector. These power similarity search, clustering, and recommendations.

We'll embed each product description so a future search service can find 'similar products.'

In [ ]:
%%sql -r dataframe_6
SELECT product_name,
       AI_EMBED(
           'snowflake-arctic-embed-m-v1.5',
           description
       )::VECTOR(FLOAT, 768) AS description_embedding
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
WHERE description IS NOT NULL
LIMIT 5;

> 🔎 **Review the output.** Each embedding is a 768-dimensional vector. Vector distance functions (and Cortex Search) use it to find semantically similar items.

### Bonus: find similar products using vector distance.

In [ ]:
%%sql -r dataframe_7
-- Find the 5 products most similar to product_id = 101 (Espresso)
WITH base AS (
    SELECT AI_EMBED('snowflake-arctic-embed-m-v1.5', description)::VECTOR(FLOAT, 768) AS emb
    FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
    WHERE product_id = 101
)
SELECT p.product_name, p.category,
       VECTOR_COSINE_SIMILARITY(
           AI_EMBED('snowflake-arctic-embed-m-v1.5', p.description)::VECTOR(FLOAT, 768),
           base.emb
       ) AS similarity
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS p, base
WHERE p.product_id != 101 AND p.description IS NOT NULL
ORDER BY similarity DESC
LIMIT 5;

## 6 · Build the Enrichment Table
Combine standardize + extract + embed into a single CTAS in **your** database — the transformation step you'd schedule in a pipeline.

> 💡 **Why a table, not a view?** A view would re-run every AI function *on each query* — paying tokens again and again. A `CREATE TABLE … AS` computes the AI once and stores the results, so downstream reads are fast and cost nothing extra. The catch is freshness: a table is a point-in-time snapshot. Step 8 shows the Snowflake-native fix — a **Dynamic Table** that keeps it current automatically.

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE {{my_db}}.RAW.PRODUCTS_ENRICHED AS
WITH extracted AS (
    SELECT
        p.product_id, p.product_name, p.category, p.unit_price, p.description,
        AI_EXTRACT(p.description, ['origin_country', 'roast_level', 'flavor_notes']) AS extract_result
    FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS p
    WHERE p.description IS NOT NULL
)
SELECT
    product_id, product_name, category, unit_price,
    description AS original_description,
    extract_result:response:origin_country::STRING AS origin_country,
    extract_result:response:roast_level::STRING   AS roast_level,
    extract_result:response:flavor_notes::STRING  AS flavor_notes,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', description)::VECTOR(FLOAT, 768) AS description_embedding
FROM extracted;

In [ ]:
%%sql -r dataframe_9
-- Verify the enriched table
SELECT product_name, origin_country, roast_level, flavor_notes
FROM {{my_db}}.RAW.PRODUCTS_ENRICHED
LIMIT 10;

## 7 · Cost Estimation With AI_COUNT_TOKENS
AI functions bill per **token** processed. A **token** is a chunk of text the model reads — very roughly **~4 characters of English**. Every call turns your text into tokens; Snowflake meters them and converts them to **credits** at a per-model rate, and credits are Snowflake's billing unit (the same ones you costed out in Lab 4). More text → more tokens → more credits.

Before scheduling this as a recurring pipeline, estimate the token budget per refresh. The query below uses [`AI_COUNT_TOKENS`](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens) to total the tokens your enrichment would consume across all products — separately for the `AI_COMPLETE` standardization and the `AI_EMBED` step, since they use different models (and bill at different rates). It only incurs a little compute, not AI tokens, so it's safe to run before you commit to the full job.

In [ ]:
%%sql -r dataframe_10
SELECT
    COUNT(*) AS total_products,
    SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', description))               AS llm_tokens_total,
    SUM(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-m-v1.5', description)) AS embed_tokens_total,
    ROUND(AVG(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', description)), 1)      AS avg_llm_tokens
FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS
WHERE description IS NOT NULL;

> 📌 **Note:** In production, only process new/changed rows. The next section shows the Snowflake-native way to do exactly that: a **Dynamic Table**.

## 8 · Operationalize With a Dynamic Table
Your §6 CTAS was a one-time snapshot. New products keep arriving — and you don't want to re-run AI on the *whole* catalog every time. The last piece is keeping the enrichment fresh **without** re-paying AI cost on rows that haven't changed.

A **[Dynamic Table](https://docs.snowflake.com/en/user-guide/dynamic-tables-about)** (you built one back in Lab 2) is the simplest way. You **define what you want once** — just the `SELECT`, with no column names or types to declare; it infers them — say how fresh to keep it with `TARGET_LAG`, and Snowflake does the rest: it re-runs your query on a schedule and applies your **AI functions only to the new or changed rows**.

> 💡 **Why this matters here:** incremental refresh means AI reprocesses *deltas only* — you pay tokens for what changed, not the whole table. That's the AI-cost payoff; the Dynamic Table is just the easy way to get it.

The shared `PRODUCTS` table is read-only, so your Dynamic Table reads from a writable copy in **your** database — that's what lets you add a row in a moment and watch the AI enrich just it.

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE {{my_db}}.RAW.PRODUCTS_SOURCE AS
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS;

🧩 **Your turn.** This lab is about AI functions first — so *you* supply the star of the show. The Dynamic Table below is complete except for the **AI function** that pulls structured fields out of each description. Fill the `______` with that function's name, then run it. Stuck? Reveal the reference below the cell.

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE DYNAMIC TABLE {{my_db}}.RAW.PRODUCTS_ENRICHED_LIVE
    TARGET_LAG = '1 hour'          -- how fresh to keep it; Snowflake schedules the refresh
    WAREHOUSE  = PLATPLUS_WH
AS
SELECT
    product_id, product_name, category, unit_price, description,
    ______(description, ['origin_country', 'roast_level', 'flavor_notes']):response::STRING AS extracted_fields   -- 🧩 name the AI function that pulls fields from text
FROM {{my_db}}.RAW.PRODUCTS_SOURCE
WHERE description IS NOT NULL;

> <details><summary>🧩 Stuck? Reference solution</summary>
>
>     AI_EXTRACT(description, ['origin_country', 'roast_level', 'flavor_notes']):response::STRING AS extracted_fields
>
> `AI_EXTRACT` is the field-pulling function from §3. And notice you never declared any columns or types for the Dynamic Table — the `SELECT` defines them for you.
> </details>

> 🔎 **That's it.** No scheduling code, no column definitions — you described the result once and set a freshness target. This is the usual workflow: prototype with a CTAS (§6), then promote the proven logic to a Dynamic Table. Now let's prove it only touches new rows.

### Watch it enrich only the new row
Add a brand-new product to your source, trigger a refresh, and see the Dynamic Table run the AI on **just that row**.

In [ ]:
%%sql -r dataframe_13
INSERT INTO {{my_db}}.RAW.PRODUCTS_SOURCE VALUES (
    109, 'Matcha Latte', 'Beverage', 5.25,
    'Ceremonial-grade matcha from Uji, Japan. Whisked with oat milk for a creamy, earthy latte '
    || 'with subtle vegetal sweetness and a smooth finish.'
);

In [ ]:
%%sql -r dataframe_14
-- TARGET_LAG would refresh this on its own; we trigger it now so we don't have to wait
ALTER DYNAMIC TABLE {{my_db}}.RAW.PRODUCTS_ENRICHED_LIVE REFRESH;

In [ ]:
%%sql -r dataframe_15
-- The new product appears, AI-enriched — and only it was processed this refresh
SELECT product_id, product_name, extracted_fields
FROM {{my_db}}.RAW.PRODUCTS_ENRICHED_LIVE
WHERE product_id = 109;

> 🔎 **The payoff.** Only that one new row was sent through `AI_EXTRACT` — the existing products weren't reprocessed. As your catalog grows from 8 to 8,000 items, each refresh pays AI tokens only for what's new. That's *AI functions, operationalized*: define the enrichment once, and let Snowflake keep it fresh at delta cost.

---
## 🤖 CoCo Assist
You've now met five AI functions. In the real world you won't always know *which* one fits a new requirement — and that's fine. This is where CoCo earns its keep: describe the goal, let it map that to the right function(s), then help you write the SQL.

Your team lead asks: *"Add a sentiment score to the feedback, and also pull out which product and issue each comment is about — then persist it as an enriched table we can schedule."*

That's actually **two** AI jobs in one — rate the tone, and pull structured fields from free text. Which functions? Don't guess — ask CoCo.

**Step 1 — ask CoCo which functions fit:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I have @PLATPLUS_DB.ROASTERY_SALES.FEEDBACK with a free-text COMMENT column. I need two things per row: the sentiment of the comment, and structured fields pulled from the text (which product it mentions, and the issue type). Which Snowflake Cortex AI functions should I use for each, and why? SQL-callable functions only. Use only functions from the current Snowflake Cortex AI Functions docs — no deprecated aliases.</pre>

> 🔎 **Read CoCo's recommendation.** It should point you to **`AI_SENTIMENT`** for the tone and **`AI_EXTRACT`** for the structured fields. Now that you know the functions, ask CoCo to build the persisted table.

**Step 2 — ask CoCo to build it:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write a CREATE OR REPLACE TABLE in my user database not PLATPLUS_DB (write to schema RAW, table FEEDBACK_ENRICHED) that adds an AI_SENTIMENT column and an AI_EXTRACT column (extracting product_mentioned and issue_type) to @PLATPLUS_DB.ROASTERY_SALES.FEEDBACK. Only rows where comment IS NOT NULL. Provide me the SQL, do not run it.</pre>

Review CoCo's output, paste it below, and run it.

In [ ]:
%%sql -r dataframe_16
-- Paste CoCo's Step 2 query on the blank line below, then run it


### Iterate — sharpen the extraction 💡
Run it and read the extracted fields. Getting nulls or vague values? AI_EXTRACT quality depends on how clearly you name the fields. Ask CoCo to improve them:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">My AI_EXTRACT is returning vague or null values for product_mentioned and issue_type. Suggest clearer field names or a short description per field to improve extraction accuracy from short customer comments.</pre>

### More ways to lean on CoCo with AI functions 🤖
CoCo isn't just for writing queries. A few prompts worth keeping in your back pocket:
- **Pick a model:** *"For standardizing product descriptions, which Cortex model balances cost and quality — and how do I pin it?"*
- **Estimate cost:** *"Estimate the token cost of running this enrichment on 100k products and suggest ways to cut it."*
- **Operationalize:** *"Convert this CTAS into a Dynamic Table with a 1-hour target lag."*

🤖 **Try it now.** Pick one of the prompts above (or invent your own) and paste it into CoCo. Review what it suggests, then paste the SQL into the cell below and run it.

> 📌 CoCo's suggestion is a starting point — you may need to iterate. Refine based on the result, just like you would with a colleague.

In [ ]:
%%sql -r dataframe_17
-- Paste CoCo's suggestion on the blank line below, then run it


## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ('RAW.PRODUCTS_ENRICHED has AI-generated descriptions', f'SELECT COUNT(*) FROM {my_db}.RAW.PRODUCTS_ENRICHED', 1),
    ('RAW.PRODUCTS_ENRICHED_LIVE dynamic table has rows', f'SELECT COUNT(*) FROM {my_db}.RAW.PRODUCTS_ENRICHED_LIVE', 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- AI_COMPLETE rewrites/standardizes inconsistent free text into a consistent format
- AI_EXTRACT pulls named fields into a structured OBJECT (values under the `response` key)
- AI_TRANSLATE normalizes multilingual data into a single language
- AI_EMBED creates 768-dim vectors for similarity search and clustering
- AI_COUNT_TOKENS lets you budget token usage before scheduling a pipeline
- CTAS = compute AI once and persist; query the enriched table cheaply
- Dynamic Tables operationalize it: define SQL + TARGET_LAG once, and AI runs incrementally on only new/changed rows

**The Other Personas:**
- The **Data Analyst** notebook scores sentiment, classifies issues, and rolls up feedback summaries with AI_AGG
- The **Platform Admin** notebook adds governance: PII redaction, content moderation, cost monitoring, and model access control

> :bulb: Each persona lab is independent — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.